### Ingestão — Brazil Journal / INFRA Journal

**O que foi construído**
Diferente do notebook de referência (Google News, que resolve descoberta via busca por query), essa fonte é única e fixa, então parte da lógica precisou ser refeita do zero:

- **Descoberta de feed:** testa em ordem o feed RSS dedicado à editoria (`/tag/infra-journal/feed/`) e cai para o feed geral do site como fallback, validando resposta HTTP e presença de itens antes de aceitar qualquer um.
- **Filtro editorial e temporal:** o Google News filtra por assunto e por data direto na busca (`query`, `when:1d`); um feed de site não oferece isso, então esse filtro (categoria + janela de 24h) foi implementado manualmente.
- **Detecção de paywall:** função nova, já que essa fonte tem escopo restrito a conteúdo público — não existia necessidade equivalente no notebook original.
- Download, limpeza de texto e persistência foram reaproveitados do notebook original.

**O que está bloqueado**
O corpo das matérias é renderizado via JavaScript, não entregue no HTML inicial da requisição — `httpx`/`curl_cffi` recebem só a casca da página (~160KB de HTML, mas só ~6.400 chars de texto visível, nenhum do artigo). A solução é Selenium (navegador headless), com infraestrutura já preparada no Volume (`/utils/selenium/`), mas ativá-la exige permissão de admin para anexar o init script ao cluster e reiniciar — fora do meu acesso atual.

In [0]:
%pip install --quiet feedparser beautifulsoup4 httpx lxml curl_cffi selenium 
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Carrega a função compartilhada que atualiza o status real da fonte na
# tabela de controle a cada execução.
%run "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/scripts/utils_controle"


In [0]:
# =============================================================================
# Imports
# =============================================================================
import os
import re
import json
import time
import random
import hashlib
import unicodedata
import urllib.parse
from datetime import datetime, timezone, timedelta
from email.utils import parsedate_to_datetime
from typing import Optional

import feedparser
import httpx
from bs4 import BeautifulSoup

# fallback nesse caso
from curl_cffi import requests as cffi_requests

In [0]:
# =============================================================================
# Configuração
# =============================================================================

# adaptada para a coleta de dados de uma source definida,
# e não mais de queries múltiplas do Google News

# Data de referência (usada no nome da pasta destino).
HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}GERAL"
os.makedirs(PASTA_DESTINO, exist_ok=True)
print(f"[setup] Salvando artefatos em: {PASTA_DESTINO}")

# Slug da tag/editoria no site. Usado tanto na URL do feed dedicado quanto no
# filtro de categoria (caso fallback)
TAG_SLUG = "infra-journal"
TAG_LABEL_HUMANO = "INFRA Journal"  # como a categoria aparece no <category> do RSS

# Candidatos de feed, em ordem de preferência. O primeiro que responder com
# itens é usado; qual foi usado é logado explicitamente (ver obter_feed_ativo).

FEED_CANDIDATOS = [
    {
        "url": f"https://braziljournal.com/tag/{TAG_SLUG}/feed/",
        "tipo": "tag_dedicada",   # não precisa filtrar por categoria depois
    },
    {
        "url": "https://braziljournal.com/feed/",
        "tipo": "geral_filtrado",  # precisa filtrar por categoria depois
    },
]

# Janela de coleta: só processamos itens publicados nas últimas N horas
JANELA_HORAS = 24

# Identificação da fonte no schema canônico de metadados.
SOURCE_ID = "brazil_journal_infra_journal"
SOURCE_DESCRICAO = "Linked from Brazil Journal — INFRA Journal"

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36 Edg/124.0.0.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/17.4 Safari/605.1.15",
]

IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124", "safari17_0", "edge101"]

HTTP_TIMEOUT = 30

# Mínimo de caracteres no texto extraído para considerar o artigo válido.
MIN_CHARS_TEXTO = 200

[setup] Salvando artefatos em: /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-31


In [0]:
# =============================================================================
# Helpers 
# ============================================================================

# função domain_from_url eliminada, já que aqui tem um source fixo e ela não faz mais sentido

def slugify(texto: str, max_len: int = 80) -> str:
    """
    Transforma uma string qualquer em um "slug" seguro pra ser usado
    como nome de arquivo (sem acento, sem espaço, sem caractere estranho).
    """
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    """Hash MD5 curto — útil pra desambiguar nomes de arquivo iguais."""
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def parsear_data_rss(data_str: str) -> str:
    """
    Converte uma string de data no formato RFC 2822 do RSS
    (ex.: 'Sun, 22 Jun 2026 12:00:00 GMT') para 'YYYY-MM-DD'.
    Devolve a data de hoje como fallback se o parse falhar.
    """
    try:
        return parsedate_to_datetime(data_str).strftime("%Y-%m-%d")
    except Exception:
        return HOJE


def parece_paywall(texto: str) -> bool:
    """
    Heurística simples: detecta marcadores comuns de paywall em sites BR.
    Retorna True se o texto parece ser conteúdo bloqueado.
    """
    marcadores = [
        "para continuar lendo",
        "assine já",
        "conteúdo exclusivo para assinantes",
        "faça login para ler",
        "este conteúdo é para assinantes",
        "cadastre-se para continuar",
    ]
    trecho = texto[:1500].lower()
    return any(m in trecho for m in marcadores)


def headers_aleatorios(referer: Optional[str] = None) -> dict:
    """Monta um dicionário de headers HTTP parecido com o de um browser real."""
    ua = random.choice(USER_AGENTS)
    headers = {
        "User-Agent": ua,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,"
                  "image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
        "Accept-Encoding": "gzip, deflate",
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Sec-Fetch-User": "?1",
        "Upgrade-Insecure-Requests": "1",
    }
    if referer:
        headers["Referer"] = referer
    return headers

In [0]:
# =============================================================================
# Etapa 1 — Verificação de feed
# =============================================================================

# primeira verificação feita antes da ingestão de fato
# criada dessa forma em consonância com o formato do site (Wordpress/ RSS de tag)

def obter_feed_ativo(candidatos: list[dict], tentativas: int = 2) -> tuple[feedparser.FeedParserDict, dict]:
    """
    Tenta cada candidato de feed em ordem. Retorna o primeiro que responde 
    HTTP 200 (sucesso no acesso a URL) E que tem pelo menos 1 resultado concreto.

    """
    for candidato in candidatos:
        url = candidato["url"]
        resp = None

        for tentativa in range(1, tentativas + 1):
            try:
                resp = httpx.get(url, timeout=15, follow_redirects=True,
                                  headers={"User-Agent": random.choice(USER_AGENTS)})
                break
            except Exception as e:
                print(f"[feed] {url} -> tentativa {tentativa}/{tentativas} falhou: {e}")
                if tentativa < tentativas:
                    time.sleep(random.uniform(1.0, 2.5))

        if resp is None:
            print(f"[feed] {url} -> todas as {tentativas} tentativas falharam, pulando candidato.")
            continue

        if resp.status_code != 200:
            print(f"[feed] {url} -> HTTP {resp.status_code}, pulando.")
            continue

        parsed = feedparser.parse(resp.content)
        n = len(parsed.entries) # n é o número de entries no feed

        if n == 0:
            print(f"[feed] {url} -> HTTP 200 mas 0 entries (feed vazio ou formato inesperado), pula.")
            continue

        print(f"[feed] OK: usando '{url}' (tipo={candidato['tipo']}), {n} entries no feed bruto.")
        return parsed, candidato

    raise RuntimeError(
        "Nenhum candidato de feed respondeu com itens. "
        "Verifique manualmente as URLs em FEED_CANDIDATOS."
    )

In [0]:
# =============================================================================
# Etapa 2 — Verificação de editoria e janela de tempo 
# =============================================================================

# segunda etapa de verificação (fallback editorial + data da publicação)

def entry_pertence_a_infra(entry, tag_label: str = TAG_LABEL_HUMANO) -> bool:
    """
    Só é chamada quando caímos no feed geral (fallback). Itens do WordPress
    trazem as tags/categorias do post em `entry.tags` (lista de objetos com
    `.term`). Comparação é case-insensitive e tolera variações de acento/caixa.
    """
    termos = [t.get("term", "") for t in entry.get("tags", [])]
    alvo = unicodedata.normalize("NFKD", tag_label).encode("ascii", "ignore").decode().lower()
    for termo in termos:
        termo_norm = unicodedata.normalize("NFKD", termo).encode("ascii", "ignore").decode().lower()
        if alvo in termo_norm:
            return True
    return False


def dentro_da_janela(entry, horas: int = JANELA_HORAS) -> bool:
    """Compara a data de publicação do item com a janela de coleta."""
    if not entry.get("published_parsed"):
        print(f"    -> aviso: item sem published_parsed ({entry.get('title', '?')[:60]}); incluindo mesmo assim.")
        return True

    pub_dt = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc)
    limite = datetime.now(timezone.utc) - timedelta(hours=horas)
    return pub_dt >= limite


def selecionar_itens(feed: feedparser.FeedParserDict, candidato: dict) -> list:
    """
    Aplica os filtros corretos dependendo de qual feed respondeu.
    Não existia antes, porque o google news automatizava isso.
    
    """
    itens = feed.entries

    if candidato["tipo"] == "geral_filtrado":
        antes = len(itens)
        itens = [e for e in itens if entry_pertence_a_infra(e)]
        print(f"[filtro] feed geral: {antes} itens -> {len(itens)} pertencem a '{TAG_LABEL_HUMANO}'.")

    antes = len(itens)
    itens = [e for e in itens if dentro_da_janela(e)]
    print(f"[filtro] janela de {JANELA_HORAS}h: {antes} itens -> {len(itens)} dentro do prazo.")

    return itens

In [0]:
# =============================================================================
# Etapa 3a — Setup do Selenium (Chrome headless via init script de cluster)
# =============================================================================
# Depende do init script estar anexado ao cluster

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, WebDriverException

CHROME_BIN = "/tmp/chrome/chrome-linux/chrome"
CHROMEDRIVER_BIN = "/tmp/chrome/chromedriver_linux64/chromedriver"

if not os.path.exists(CHROME_BIN) or not os.path.exists(CHROMEDRIVER_BIN):
    raise RuntimeError(
        "Chrome/chromedriver não encontrados em /tmp/chrome. "
        "O init script provavelmente não está anexado a este cluster. "
        "Verifique em: Compute > [seu cluster] > Configuration > "
        "Advanced options > Init Scripts."
    )

os.chmod(CHROME_BIN, 0o755)
os.chmod(CHROMEDRIVER_BIN, 0o755)


def criar_driver() -> webdriver.Chrome:
    options = webdriver.ChromeOptions()
    options.binary_location = CHROME_BIN
    options.add_argument('headless')
    options.add_argument('--disable-infobars')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--no-sandbox')
    options.add_argument('--remote-debugging-port=9222')
    options.add_argument('--homedir=/tmp/chrome/chrome-user-data-dir')
    options.add_argument('--user-data-dir=/tmp/chrome/chrome-user-data-dir')
    options.add_argument(f"--user-agent={random.choice(USER_AGENTS)}")
    prefs = {
        "download.default_directory": "/tmp/chrome/chrome-user-data-dir",
        "download.prompt_for_download": False,
    }
    options.add_experimental_option("prefs", prefs)

    service = Service(CHROMEDRIVER_BIN)
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(30)
    return driver

In [0]:
# =============================================================================
# Etapa 3 — Download do HTML da matéria (via navegador headless)
# =============================================================================

def baixar_html(url: str, driver, tentativas: int = 2) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        try:
            driver.get(url)
            # Espera até 10s por sinal de que o conteúdo real carregou.
            WebDriverWait(driver, 10).until(
                lambda d: len(d.find_element(By.TAG_NAME, "body").text) > 500
            )
            time.sleep(1)  # margem extra pra scripts tardios
            return driver.page_source
        except TimeoutException:
            print(f"    -> tentativa {tentativa}/{tentativas}: timeout esperando conteúdo carregar.")
        except WebDriverException as e:
            print(f"    -> tentativa {tentativa}/{tentativas}: erro do driver ({e}).")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.0))

    return None

In [0]:
# =============================================================================
# Etapa 4 — Limpar o HTML e extrair só o texto útil
# =============================================================================

# Tags cujo conteúdo raramente serve pra análise textual.
TAGS_LIXO = [
    "script", "style", "noscript", "iframe", "svg", "form",
    "nav", "footer", "header", "aside", "button",
]

def extrair_texto(html: str) -> str:
    if not html:
        return ""

    soup = BeautifulSoup(html, "lxml")

    for tag in soup(TAGS_LIXO):
        tag.decompose()

    article = soup.find("article")
    if article and len(article.get_text(strip=True)) > 500:
        base = article
    else:
        base = soup  # fallback: <article> não confiável nesse tema, usa a página inteira

    texto = base.get_text("\n", strip=True)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()

In [0]:
# =============================================================================
# Etapa 5 — Salvar no Volume
# =============================================================================

def salvar_artefatos(
    pasta: str,
    source: str,
    titulo: str,
    texto: str,
    metadados: dict,
) -> tuple[str, str]:
    """
    Persiste no Volume os dois arquivos por matéria:
      - {source}_{article_name}.txt  (texto limpo)
      - {source}_{article_name}.json (metadados)
    """
    slug_source = slugify(source, max_len=40) or "fonte"
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")

    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json

In [0]:
# =============================================================================
# Etapa 6 — Pipeline principal
# =============================================================================

def processar_entry(entry, driver) -> Optional[dict]:
    titulo = entry.get("title", "sem-titulo")
    url = entry.get("link")
    print(f"\n  [item] {titulo[:100]}")

    if not url:
        print("    -> sem link; pulando.")
        return None

    html = baixar_html(url, driver)
    if not html:
        print("    -> download do HTML falhou; pulando.")
        return None

    texto = extrair_texto(html)

    if parece_paywall(texto):
        print("    -> marcador de paywall detectado; pulando (escopo = só conteúdo público).")
        return None

    if not texto or len(texto) < MIN_CHARS_TEXTO:
        print(f"    -> texto muito curto ({len(texto)} chars); pulando.")
        return None

    if entry.get("published_parsed"):
        data_publicacao = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc).isoformat()
    else:
        data_publicacao = None

    # Monta metadados no padrão canônico de ingestão:
    # source_id / title / description / url / date / published_at
    metadados = {
        "source_id": SOURCE_ID,
        "title": titulo,
        "description": SOURCE_DESCRICAO,
        "url": url,
        "date": HOJE,
        "published_at": data_publicacao,
    }

    caminho_txt, caminho_json = salvar_artefatos(
        pasta=PASTA_DESTINO,
        source=SOURCE_ID,
        titulo=titulo,
        texto=texto,
        metadados=metadados,
    )
    print(f"    -> salvo em {caminho_txt}")

    return {
        "titulo": titulo,
        "url": url,
        "caminho_txt": caminho_txt,
        "caminho_json": caminho_json,
    }

In [0]:
# =============================================================================
# Execução
# =============================================================================

try:
    feed, candidato_usado = obter_feed_ativo(FEED_CANDIDATOS)
    itens = selecionar_itens(feed, candidato_usado)

    print(f"\n=== Feed usado: {candidato_usado['url']} (tipo={candidato_usado['tipo']}) ===")
    print(f"=== {len(itens)} itens a processar ===")

    # criar_driver() é o ponto que falha quando o init script do Selenium
    # não está anexado ao cluster (bloqueio conhecido, ver aba Bloqueios) —
    # por isso está dentro do try, não antes dele.
    driver = criar_driver()
    todos_resultados: list[dict] = []

    try:
        for entry in itens:
            try:
                resultado = processar_entry(entry, driver)
                if resultado:
                    todos_resultados.append(resultado)
            except Exception as e:
                print(f"[ERRO] item {entry.get('title', '?')!r} falhou: {e}")
    finally:
        driver.quit()

    print(f"\n\n=== Fim. {len(todos_resultados)} matérias salvas em {PASTA_DESTINO} ===")
    print(f"=== Fonte de feed que respondeu nesta execução: {candidato_usado['url']} ===")

    atualizar_status_fonte(
        source_id=SOURCE_ID,
        sucesso=True,
        docs_capturados=len(todos_resultados),
    )

except Exception as e:
    print(f"\n\n=== ERRO GERAL: {e} ===")
    atualizar_status_fonte(
        source_id=SOURCE_ID,
        sucesso=False,
        docs_capturados=0,
        erro=str(e),
    )

[feed] OK: usando 'https://braziljournal.com/tag/infra-journal/feed/' (tipo=tag_dedicada), 10 entries no feed bruto.
[filtro] janela de 24h: 10 itens -> 1 dentro do prazo.

=== Feed usado: https://braziljournal.com/tag/infra-journal/feed/ (tipo=tag_dedicada) ===
=== 1 itens a processar ===

  [item] Contratos do leilão de capacidade viram disputa na Aneel. Quem vai levar?
    -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-31/brazil-journal-infra-journal_contratos-do-leilao-de-capacidade-viram-disputa-na-aneel-que_e61a430b.txt


=== Fim. 1 matérias salvas em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-31 ===
=== Fonte de feed que respondeu nesta execução: https://braziljournal.com/tag/infra-journal/feed/ ===
